In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE


In [2]:

df = pd.read_csv('final_train.csv')
df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 213451 entries, 0 to 213450
Data columns (total 29 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       213451 non-null  str    
 1   gender                   213451 non-null  str    
 2   age                      213451 non-null  float64
 3   signup_method            213451 non-null  str    
 4   signup_flow              213451 non-null  str    
 5   language                 213451 non-null  str    
 6   affiliate_channel        213451 non-null  str    
 7   affiliate_provider       213451 non-null  str    
 8   first_affiliate_tracked  213451 non-null  str    
 9   signup_app               213451 non-null  str    
 10  first_browser            213451 non-null  str    
 11  country_destination      213451 non-null  str    
 12  is_train                 213451 non-null  bool   
 13  total_actions            213451 non-null  float64
 14  total_seconds  

In [3]:
# Separate Features (X) and Target (y)
X = df.drop(['id', 'country_destination' , 'is_train'], axis=1)
y = df['country_destination']


# Scikit-Learn requires the target to be numbers (0 to 11), not strings ('US', 'FR')
le = LabelEncoder()
y_encoded = le.fit_transform(y)


# Save the mapping so we know which number is which country later!
target_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Target Mapping: {target_mapping}")


Target Mapping: {'AU': np.int64(0), 'CA': np.int64(1), 'DE': np.int64(2), 'ES': np.int64(3), 'FR': np.int64(4), 'GB': np.int64(5), 'IT': np.int64(6), 'NDF': np.int64(7), 'NL': np.int64(8), 'PT': np.int64(9), 'US': np.int64(10), 'other': np.int64(11)}


In [4]:
numerical_cols = X.select_dtypes(include=['int64', 'float64' ]).columns
categorical_cols = X.select_dtypes(include=['object' , 'str']).columns 

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

((170760, 26), (42691, 26), (170760,), (42691,))

In [5]:


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0))
])


categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [6]:
import numpy as np
def calculate_ndcg5(y_true, y_pred_proba):
    """
    Calculates the exact NDCG@5 score used by Kaggle.
    y_true: 1D array of true encoded labels
    y_pred_proba: 2D array of XGBoost probabilities
    """
    # Get the column index (class) of the top 5 highest probabilities per user
    top5_preds = np.argsort(y_pred_proba, axis=1)[:, :-6:-1]
    
    # Check which of the 5 positions matches the true label
    matches = (top5_preds == y_true.reshape(-1, 1))
    
    # Apply the discount math: 1 / log2(rank + 1)
    discounts = 1 / np.log2(np.arange(1, 6) + 1)
    
    # Multiply matches by discounts and get the average score across all users
    ndcg_scores = np.sum(matches * discounts, axis=1)
    return np.mean(ndcg_scores)

In [ ]:
master_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor) , 
    ('model' , XGBClassifier(
    n_estimators=300,       
    learning_rate=0.1,      
    max_depth=6,           
    subsample=0.8,          
    colsample_bytree=0.8,    
    objective='multi:softprob', 
    random_state=42,
    n_jobs=-1
))
    ]) 

In [7]:
x_preprocessed = preprocessor.fit_transform(X) 

### xgboost with train_test_split 

In [29]:
master_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [31]:
y_train_proba=master_pipeline.predict_proba(X_train)
train_ndcg = calculate_ndcg5(y_train, y_train_proba)
print(f"xgb Train NDCG@5: {train_ndcg:.5f}")

xgb Train NDCG@5: 0.85384


In [34]:
test_df = pd.read_csv('final_test.csv')

# Separate the IDs (we need these for Kaggle) and the Features
test_ids = test_df['id']
X_test = test_df.drop('id', axis=1)

# Pass the raw test data straight into the pipeline!
# It will automatically clean the data, encode the categories, and predict.
y_test_proba = master_pipeline.predict_proba(X_test)


# Get the column index of the top 5 highest probabilities for every single user
top5_indices = np.argsort(y_test_proba, axis=1)[:, :-6:-1]

# Use our LabelEncoder (le) from earlier to translate the math numbers (0-11) 
# back into the actual country strings ('US', 'FR', 'NDF')
top5_countries = le.inverse_transform(top5_indices.flatten()).reshape(top5_indices.shape)


# Kaggle requires the data to be stacked (5 rows per user)
# np.repeat duplicates each user's ID 5 times perfectly
submission_ids = np.repeat(test_ids.values, 5)

# .flatten() unravels our top 5 matrix into a single long column
submission_countries = top5_countries.flatten()

# Build the final dataframe
submission_df = pd.DataFrame({
    'id': submission_ids,
    'country': submission_countries
})

# Save it! 
submission_df.to_csv('baseline_xgb_submission.csv', index=False)

Note -> This model get **0.87880** on kaggle        

#### xgboost using **Optuna AI** with 15 trial 

In [ ]:
import optuna

print("1. Defining the Optuna AI...")

def objective(trial):
    
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        
        # Static parameters 
        'objective': 'multi:softprob',
        'random_state': 42,
        'n_jobs': -1 
    }
    
    # 2. Build a new XGBoost model with Optuna's suggested parameters
    xgb_tuned = XGBClassifier(**param)
    
    # 3. Swap the old model 
    master_pipeline.steps[-1] = ('model', xgb_tuned)
    
  
    master_pipeline.fit(X_train, y_train)
    
   
    y_val_proba = master_pipeline.predict_proba(X_val)
    

    score = calculate_ndcg5(y_val, y_val_proba)
    
 
    return score




study = optuna.create_study(direction='maximize')

# Run 30 (trials). 

study.optimize(objective, n_trials=15)

print("\nOptimization Complete!")
print("Best NDCG@5 Score:", study.best_value)
print("Best Hyperparameters:", study.best_params)

[I 2026-04-05 01:31:15,728] A new study created in memory with name: no-name-e1f7882a-8d79-4846-8620-b3dd75e4c793


1. Defining the Optuna AI...
2. Launching the Hyperparameter Hunt...


[I 2026-04-05 01:31:57,888] Trial 0 finished with value: 0.8300027528522208 and parameters: {'n_estimators': 481, 'max_depth': 4, 'learning_rate': 0.0340794218165568, 'subsample': 0.9967282035867693, 'colsample_bytree': 0.7439236922541117, 'min_child_weight': 5}. Best is trial 0 with value: 0.8300027528522208.
[I 2026-04-05 01:33:02,692] Trial 1 finished with value: 0.8304569838077634 and parameters: {'n_estimators': 581, 'max_depth': 6, 'learning_rate': 0.03191746896318677, 'subsample': 0.6791057104903218, 'colsample_bytree': 0.8348210256386005, 'min_child_weight': 3}. Best is trial 1 with value: 0.8304569838077634.
[I 2026-04-05 01:33:49,454] Trial 2 finished with value: 0.8234972023673379 and parameters: {'n_estimators': 288, 'max_depth': 10, 'learning_rate': 0.1866153008595972, 'subsample': 0.8545388758306325, 'colsample_bytree': 0.8918793214882073, 'min_child_weight': 2}. Best is trial 1 with value: 0.8304569838077634.
[I 2026-04-05 01:34:26,762] Trial 3 finished with value: 0.830


🏆 Optimization Complete!
Best NDCG@5 Score: 0.8306031739214206
Best Hyperparameters: {'n_estimators': 568, 'max_depth': 8, 'learning_rate': 0.0185348680625659, 'subsample': 0.5917729035263337, 'colsample_bytree': 0.6522032753663959, 'min_child_weight': 7}


Bestmodel 

In [ ]:

# XGBoost parameters tuned for this specific dataset
xgb_model = XGBClassifier(
    n_estimators=568,        
    learning_rate=0.0185348680625659,      
    max_depth=8,             
    subsample=0.5917729035263337,          
    colsample_bytree=0.6522032753663959,   
    objective='multi:softprob', 
    random_state=42,
    n_jobs=-1 , 
    min_child_weight=7
)


master_pipeline.steps[-1] = ('model', xgb_model)


master_pipeline.fit(X_train, y_train)
y_train_proba = master_pipeline.predict_proba(X_train)
train_score=calculate_ndcg5(y_train, y_train_proba)

print(f"\nTrain NDCG@5 Score: {train_score:.5f}")


y_val_proba = master_pipeline.predict_proba(X_val)


test_score = calculate_ndcg5(y_val, y_val_proba)

print(f"\nTest NDCG@5 Score: {test_score:.5f}")


Train NDCG@5 Score: 0.84526

Test NDCG@5 Score: 0.83060


In [38]:
test_df = pd.read_csv('final_test.csv')

# Separate the IDs (we need these for Kaggle) and the Features
test_ids = test_df['id']
X_test = test_df.drop('id', axis=1)

# Pass the raw test data straight into the pipeline!
# It will automatically clean the data, encode the categories, and predict.
y_test_proba = master_pipeline.predict_proba(X_test)


# Get the column index of the top 5 highest probabilities for every single user
top5_indices = np.argsort(y_test_proba, axis=1)[:, :-6:-1]

# Use our LabelEncoder (le) from earlier to translate the math numbers (0-11) 
# back into the actual country strings ('US', 'FR', 'NDF')
top5_countries = le.inverse_transform(top5_indices.flatten()).reshape(top5_indices.shape)


# Kaggle requires the data to be stacked (5 rows per user)
# np.repeat duplicates each user's ID 5 times perfectly
submission_ids = np.repeat(test_ids.values, 5)

# .flatten() unravels our top 5 matrix into a single long column
submission_countries = top5_countries.flatten()

# Build the final dataframe
submission_df = pd.DataFrame({
    'id': submission_ids,
    'country': submission_countries
})

# Save it! 
submission_df.to_csv('bestmodel_xgb_optuna_submission.csv', index=False)

Note -> This model get **0.87989** on kaggle         

#### xgboost using **Optuna AI** with Stratified-kfold 

In [39]:
x_preprocessed = preprocessor.fit_transform(X) 


In [ ]:
from sklearn.model_selection import StratifiedKFold

def objective(trial):

    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 4, 8),         
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        
        # Mandatory static parameters
        'objective': 'multi:softprob',
        'eval_metric': 'mlogloss',
        'tree_method': 'hist',  
        'n_jobs': -1,          
        'random_state': 42
    
    }
    
    xgb_model = XGBClassifier(**param)
    
 
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []
    

    for train_idx, val_idx in skf.split(x_preprocessed, y_encoded):
        X_fold_train, X_fold_val = x_preprocessed[train_idx], x_preprocessed[val_idx]
        y_fold_train, y_fold_val = y_encoded[train_idx], y_encoded[val_idx]
        
        xgb_model.fit(X_fold_train, y_fold_train)
        
        y_fold_proba = xgb_model.predict_proba(X_fold_val)
        score = calculate_ndcg5(y_fold_val, y_fold_proba)
        fold_scores.append(score)
    
    return np.mean(fold_scores)


study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=10)


print(f"Highest CV NDCG@5: {study.best_value:.5f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

[I 2026-04-05 02:06:46,813] A new study created in memory with name: no-name-78398531-0595-4116-94ac-c86eb55e58eb
[I 2026-04-05 02:08:02,916] Trial 0 finished with value: 0.8302625863374012 and parameters: {'n_estimators': 364, 'max_depth': 4, 'learning_rate': 0.023156412495823134, 'subsample': 0.9984857753893663, 'colsample_bytree': 0.8097704063146168, 'min_child_weight': 3}. Best is trial 0 with value: 0.8302625863374012.
[I 2026-04-05 02:09:37,723] Trial 1 finished with value: 0.827026491292539 and parameters: {'n_estimators': 352, 'max_depth': 8, 'learning_rate': 0.1486399088310902, 'subsample': 0.7683440278108871, 'colsample_bytree': 0.7334891621681789, 'min_child_weight': 8}. Best is trial 0 with value: 0.8302625863374012.
[I 2026-04-05 02:10:44,939] Trial 2 finished with value: 0.8305532921309009 and parameters: {'n_estimators': 275, 'max_depth': 6, 'learning_rate': 0.09772536585561732, 'subsample': 0.6367519758739695, 'colsample_bytree': 0.7712260379544656, 'min_child_weight': 

Highest CV NDCG@5: 0.83088
Best Parameters:
    n_estimators: 321
    max_depth: 4
    learning_rate: 0.07239221957627491
    subsample: 0.7095705997700336
    colsample_bytree: 0.6982867704692531
    min_child_weight: 5


#### bestmodel


In [ ]:

xgb_model = XGBClassifier(
    n_estimators=321,       
    learning_rate=0.07239221957627491,    
    max_depth=4,          
    subsample=0.7095705997700336,          
    colsample_bytree=0.6982867704692531,   
    objective='multi:softprob',
    random_state=42,
    n_jobs=-1 ,  
    min_child_weight=5
)


master_pipeline.steps[-1] = ('model', xgb_model)


master_pipeline.fit(X_train, y_train)
y_train_proba = master_pipeline.predict_proba(X_train)
train_score=calculate_ndcg5(y_train, y_train_proba)

print(f"\nTrain NDCG@5 Score: {train_score:.5f}")


y_val_proba = master_pipeline.predict_proba(X_val)


test_score = calculate_ndcg5(y_val, y_val_proba)

print(f"\nTest NDCG@5 Score: {test_score:.5f}")


Train NDCG@5 Score: 0.83708

Test NDCG@5 Score: 0.83040


In [42]:
test_df = pd.read_csv('final_test.csv')

# Separate the IDs (we need these for Kaggle) and the Features
test_ids = test_df['id']
X_test = test_df.drop('id', axis=1)

# Pass the raw test data straight into the pipeline!
# It will automatically clean the data, encode the categories, and predict.
y_test_proba = master_pipeline.predict_proba(X_test)


# Get the column index of the top 5 highest probabilities for every single user
top5_indices = np.argsort(y_test_proba, axis=1)[:, :-6:-1]

# Use our LabelEncoder (le) from earlier to translate the math numbers (0-11) 
# back into the actual country strings ('US', 'FR', 'NDF')
top5_countries = le.inverse_transform(top5_indices.flatten()).reshape(top5_indices.shape)


# Kaggle requires the data to be stacked (5 rows per user)
# np.repeat duplicates each user's ID 5 times perfectly
submission_ids = np.repeat(test_ids.values, 5)

# .flatten() unravels our top 5 matrix into a single long column
submission_countries = top5_countries.flatten()

# Build the final dataframe
submission_df = pd.DataFrame({
    'id': submission_ids,
    'country': submission_countries
})

# Save it! 
submission_df.to_csv('bestmodel_xgb_optuna_stratified_submission.csv', index=False)

Note -> This model get **0.879994** on kaggle         

#### Save model and pipeline for deployment 

In [44]:
import joblib

# Save your ColumnTransformer/Pipeline using joblib
# This ensures that when a new user signs up in production, 
# their data is One-Hot Encoded the exact same way as the training data.
joblib.dump(preprocessor, 'airbnb_preprocessor.joblib')


# PRO TIP: Do not use joblib or pickle to save XGBoost models.
# Pickles are a massive security risk in production. 
# Use XGBoost's native JSON saver. It is faster, safer, and allows 
# software engineers to load the model in C++ or Java backends!
# Assuming 'best_xgb' is your final trained model variable
xgb_model.save_model('airbnb_xgboost_model.json')

joblib.dump(xgb_model, 'airbnb_xgboost_model.joblib')

['airbnb_xgboost_model.joblib']

#### xgboost with SMOTE 

In [ ]:
from sklearn.model_selection import StratifiedKFold
import optuna
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import numpy as np

def objective(trial):
   n
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200),
        'max_depth': trial.suggest_int('max_depth', 4, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        
        # Mandatory static parameters
        'objective': 'multi:softprob',
        'eval_metric': 'mlogloss',
        'tree_method': 'hist', 
        'n_jobs': -1,           
        'random_state': 42
    }
    
    xgb_model = xgb.XGBClassifier(**param)
    smote = SMOTE(random_state=42)
    
    pipeline = Pipeline(steps=[
        ('smote', smote),
        ('model', xgb_model)
    ])
    
    # 3-Fold CV
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []
    
   
    X_array = x_preprocessed.values if hasattr(x_preprocessed, 'values') else x_preprocessed
    y_array = y_encoded.values if hasattr(y_encoded, 'values') else y_encoded
    
    # K-Fold Loop
    for train_idx, val_idx in skf.split(X_array, y_array):
        X_fold_train, X_fold_val = X_array[train_idx], X_array[val_idx]
        y_fold_train, y_fold_val = y_array[train_idx], y_array[val_idx]
        
        pipeline.fit(X_fold_train, y_fold_train)
        
      
        y_fold_proba = pipeline.predict_proba(X_fold_val)
        
        # Calculate custom score
        score = calculate_ndcg5(y_fold_val, y_fold_proba)
        fold_scores.append(score)
    
    return np.mean(fold_scores)



study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print(f"Highest CV NDCG@5: {study.best_value:.5f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

# --- FINAL TRAINING ---
# Apply SMOTE to the FULL training set once
smote_final = SMOTE(random_state=42)
X_resampled, y_resampled = smote_final.fit_resample(x_preprocessed, y_encoded)

# Train the final model using Optuna's best parameters
final_xgb = xgb.XGBClassifier(**study.best_params, random_state=42)
final_xgb.fit(X_resampled, y_resampled)


final_xgb.save_model("airbnb_xgboost_smote_model.json")

[I 2026-04-05 16:52:24,732] A new study created in memory with name: no-name-a6e00097-83c4-4a6a-93dd-c273ff28c76d
[I 2026-04-05 17:01:56,910] Trial 0 finished with value: 0.8231995066461845 and parameters: {'n_estimators': 180, 'max_depth': 4, 'learning_rate': 0.16316289995548436, 'subsample': 0.791915008128594, 'colsample_bytree': 0.9773580801069226, 'min_child_weight': 9}. Best is trial 0 with value: 0.8231995066461845.
[I 2026-04-05 17:11:39,827] Trial 1 finished with value: 0.8230789252482413 and parameters: {'n_estimators': 183, 'max_depth': 5, 'learning_rate': 0.09152801669492053, 'subsample': 0.9429290603344602, 'colsample_bytree': 0.7491547577297375, 'min_child_weight': 1}. Best is trial 0 with value: 0.8231995066461845.
[I 2026-04-05 17:20:46,482] Trial 2 finished with value: 0.8241419551276447 and parameters: {'n_estimators': 152, 'max_depth': 6, 'learning_rate': 0.11601742017692061, 'subsample': 0.9496412931545851, 'colsample_bytree': 0.755533761808701, 'min_child_weight': 3

Highest CV NDCG@5: 0.82444
Best Parameters:
    n_estimators: 141
    max_depth: 6
    learning_rate: 0.15761822057417454
    subsample: 0.9714563138618386
    colsample_bytree: 0.6272657595423194
    min_child_weight: 6


In [11]:
import pandas as pd
import numpy as np

test_df = pd.read_csv('final_test.csv')

# 1. Separate the IDs (we need these for Kaggle) and the Features
test_ids = test_df['id']
X_test = test_df.drop('id', axis=1)

# 🚨 CRITICAL FIX 1 & 2: Use .transform() only, and pass X_test!
x_test_preprocessed = preprocessor.transform(X_test)   

# 2. Pass the cleaned test data into the trained model
y_test_proba = final_xgb.predict_proba(x_test_preprocessed)

# 3. Get the column index of the top 5 highest probabilities for every single user
top5_indices = np.argsort(y_test_proba, axis=1)[:, :-6:-1]

# 4. Translate the math numbers (0-11) back into actual country strings
top5_countries = le.inverse_transform(top5_indices.flatten()).reshape(top5_indices.shape)

# 5. Format for Kaggle (Stacking 5 rows per user)
submission_ids = np.repeat(test_ids.values, 5)
submission_countries = top5_countries.flatten()

# 6. Build the final dataframe
submission_df = pd.DataFrame({
    'id': submission_ids,
    'country': submission_countries
})

# 7. Save it! 
submission_df.to_csv('bestmodel_xgb_optuna_smote_submission.csv', index=False)

Note -> it get 0.87378 on kaggle

In [13]:

y_train_proba = final_xgb.predict_proba(x_preprocessed)
train_score = calculate_ndcg5(y_encoded, y_train_proba)


test_score = study.best_value

# 3. DISPLAY RESULTS
print(f"Train NDCG@5 Score: {train_score:.5f}")
print(f"Test NDCG@5 Score: {test_score:.5f}")


Train NDCG@5 Score: 0.83973
Test NDCG@5 Score: 0.82444


In [12]:
import joblib

joblib.dump(final_xgb, 'final_xgboost_model.joblib')

['final_xgboost_model.joblib']